# 앙상블

앙상블은 여러 모델의 예측을 결합해 더 안정적이고 강한 예측을 만들려는 방법이다.

1. 모델 하나가 항상 최고인 것은 아니다.
2. 서로 다른 성향의 모델을 함께 쓰면 한 모델의 약점을 다른 모델이 보완할 수 있다.
3. 앙상블의 목적은 **무조건 복잡하게 만들기**가 아니라 **예측을 더 안정적으로 만들기**이다.

대표적인 앙상블 계열은 다음과 같다.

1. Voting
   - 여러 모델의 예측을 모아 최종 결과를 결정한다.
   - 분류에서는 hard voting, soft voting을 주로 사용한다.
2. Bagging
   - 같은 계열 모델을 여러 번 학습시켜 평균/투표로 예측한다.
   - 대표적으로 Random Forest가 있다.
3. Boosting
   - 이전 모델이 잘 못 맞춘 데이터에 점점 더 집중하면서 순차적으로 학습한다.
   - 대표적으로 AdaBoost, Gradient Boosting, XGBoost 계열이 있다.
4. Stacking
   - 여러 모델의 예측값을 다시 입력값처럼 사용해 최종 메타 모델이 학습한다.

가장 직관적인 Voting부터 시작한다.

In [ ]:
import lightgbm

lightgbm.__version__

'4.6.0'

# Voting 계열

## VotingClassifier
- hard voting: 각 모델의 최종 예측 클래스를 다수결로 결정한다.
- soft voting: 각 모델의 예측 확률을 평균내어 최종 클래스를 결정한다.

특징
- hard voting은 이해하기 쉽지만, 모델의 확신 정도는 반영하지 못한다.
- soft voting은 확률 정보를 활용하므로 보통 더 유연하다.
- 단, soft voting을 쓰려면 각 모델이 `predict_proba()`를 지원해야 한다.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 01. VotingClassifier 

성향이 다른 3개 모델을 묶어 본다.

1. LogisticRegression: 선형 기반, 확률 해석이 비교적 자연스럽다.
2. KNeighborsClassifier: 거리 기반, 주변 샘플의 영향을 많이 받는다.
3. DecisionTreeClassifier: 규칙 분할 기반, 해석이 직관적이다.

서로 다른 방식의 모델을 묶는 이유는 **서로 다른 관점** 을 결합하기 위해서이다.

In [2]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 유방암 데이터 로드
cancer = load_breast_cancer()
X = cancer.data
y = cancer.target

# 훈련/테스트 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 스케일링
# KNN, LogisticRegression은 스케일의 영향을 많이 받는다.
# DecisionTree는 스케일링이 꼭 필요하지는 않지만, 하나의 입력 데이터를 여러 모델이 함께 쓰므로 여기서는 통일해서 전처리한다.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(X_train_scaled.shape, y_train.shape)
print(X_test_scaled.shape, y_test.shape)

(455, 30) (455,)
(114, 30) (114,)


### hard voting

hard voting은 각 모델이 예측한 최종 클래스를 가지고 다수결을 수행한다.

예를 들어,
- lr -> 1
- knn -> 1
- dt -> 0

이라면 최종 예측은 1이다.

즉, **누가 얼마나 확신했는가** 보다 **몇 표를 받았는가** 가 중요하다.

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import VotingClassifier

# 개별 분류기 준비
lr_clf = LogisticRegression(random_state=42, max_iter=1000)
knn_clf = KNeighborsClassifier(n_neighbors=5)
dt_clf = DecisionTreeClassifier(random_state=42)

# hard voting 분류기 준비
hard_voting_clf = VotingClassifier(
    estimators=[
        ('lr', lr_clf), 
        ('knn', knn_clf), 
        ('dt', dt_clf)
    ],
    voting='hard'
)

# 앙상블 학습
hard_voting_clf.fit(X_train_scaled, y_train)

print('hard voting 학습 정확도:', hard_voting_clf.score(X_train_scaled, y_train))
print('hard voting 평가 정확도:', hard_voting_clf.score(X_test_scaled, y_test))

hard voting 학습 정확도: 0.9912087912087912
hard voting 평가 정확도: 0.9824561403508771


In [6]:
from sklearn.metrics import accuracy_score
# 개별 분류기와 hard voting 성능 비교

# 개별 분류기 학습
lr_clf.fit(X_train_scaled, y_train)
knn_clf.fit(X_train_scaled, y_train)
dt_clf.fit(X_train_scaled, y_train)

# 개별 분류기와 하트 보팅 분류기의 예측 결과
lr_pred = lr_clf.predict(X_test_scaled)
knn_pred = knn_clf.predict(X_test_scaled)
dt_pred = dt_clf.predict(X_test_scaled)
hard_voting_pred = hard_voting_clf.predict(X_test_scaled)

# 정확도 비교
compare_df = pd.DataFrame({
    'model' : ['LogisticRegression', 'KNeighborsClassifier', 'DecisionTreeClassifier', 'HardVoting'],
    'accuracy': [
        accuracy_score(y_test, lr_pred),
        accuracy_score(y_test, knn_pred),
        accuracy_score(y_test, dt_pred),
        accuracy_score(y_test, hard_voting_pred),
    ]
})

compare_df.sort_values('accuracy', ascending=False)

,model,accuracy
0,LogisticRegression,0.982456
3,HardVoting,0.982456
1,KNeighborsClassifier,0.956140
2,DecisionTreeClassifier,0.912281


In [8]:
# 일부 샘플에 대해 각 모델이 어떻게 예측 했는지 확인
start = 10
end = 20

pred_df = pd.DataFrame([
    y_test[start:end],
    lr_pred[start:end],
    knn_pred[start:end],
    dt_pred[start:end],
    hard_voting_pred[start:end]
], index=['true', 'lr', 'knn', 'dt', 'hard voting'])

pred_df

,0,1,2,3,4,5,6,7,8,9
true,1,0,1,0,0,1,1,1,1,1
lr,1,0,1,0,0,1,0,1,1,1
knn,1,0,1,0,0,1,1,1,1,1
dt,1,0,1,0,0,1,1,1,1,1
hard voting,1,0,1,0,0,1,1,1,1,1


### soft voting

soft voting은 각 모델의 예측 확률을 평균내고, 그 평균 확률이 더 큰 클래스를 최종 예측으로 선택한다.

예를 들어 양성 클래스(1)에 대한 확률이
- lr -> 0.90
- knn -> 0.60
- dt -> 0.00

이라면 평균은 `(0.90 + 0.60 + 0.00) / 3 = 0.50` 이다.

즉, soft voting은 다수결뿐 아니라 각 모델의 확신 정도도 함께 반영한다.

In [9]:
# soft voting은 predict_proba 함수를 지원하는 모델로 구성해야 한다.
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import VotingClassifier

# 개별 분류기 준비
lr_clf = LogisticRegression(random_state=42, max_iter=1000)
knn_clf = KNeighborsClassifier(n_neighbors=5)
dt_clf = DecisionTreeClassifier(random_state=42)

# soft voting 분류기 준비
soft_voting_clf = VotingClassifier(
    estimators=[
        ('lr', lr_clf), 
        ('knn', knn_clf), 
        ('dt', dt_clf)
    ],
    voting='soft'
)

# 앙상블 학습
soft_voting_clf.fit(X_train_scaled, y_train)

print('soft voting 학습 정확도:', soft_voting_clf.score(X_train_scaled, y_train))
print('soft voting 평가 정확도:', soft_voting_clf.score(X_test_scaled, y_test))

soft voting 학습 정확도: 0.9956043956043956
soft voting 평가 정확도: 0.9649122807017544


In [10]:
from sklearn.metrics import accuracy_score
# 개별 분류기와 soft voting 성능 비교

# 개별 분류기 학습
lr_clf.fit(X_train_scaled, y_train)
knn_clf.fit(X_train_scaled, y_train)
dt_clf.fit(X_train_scaled, y_train)

# 개별 분류기와 소프트 보팅 분류기의 예측 결과
lr_pred = lr_clf.predict(X_test_scaled)
knn_pred = knn_clf.predict(X_test_scaled)
dt_pred = dt_clf.predict(X_test_scaled)
soft_voting_pred = soft_voting_clf.predict(X_test_scaled)

# 정확도 비교
compare_df = pd.DataFrame({
    'model' : ['LogisticRegression', 'KNeighborsClassifier', 'DecisionTreeClassifier', 'SoftVoting'],
    'accuracy': [
        accuracy_score(y_test, lr_pred),
        accuracy_score(y_test, knn_pred),
        accuracy_score(y_test, dt_pred),
        accuracy_score(y_test, soft_voting_pred),
    ]
})

compare_df.sort_values('accuracy', ascending=False)

,model,accuracy
0,LogisticRegression,0.982456
3,SoftVoting,0.964912
1,KNeighborsClassifier,0.956140
2,DecisionTreeClassifier,0.912281


In [11]:
# 일부 샘플에 대해 각 모델이 어떻게 예측 했는지 확인
start = 10
end = 20

# 각 모델의 예측 확률
lr_pred_proba = lr_clf.predict_proba(X_test_scaled)
knn_pred_proba = knn_clf.predict_proba(X_test_scaled)
dt_pred_proba = dt_clf.predict_proba(X_test_scaled)
soft_voting_pred_proba = soft_voting_clf.predict_proba(X_test_scaled)

pred_df = pd.DataFrame([
    y_test[start:end],
    lr_pred_proba[start:end, 1],
    knn_pred_proba[start:end, 1],
    dt_pred_proba[start:end, 1],
    soft_voting_pred_proba[start:end, 1]
], index=['true', 'lr', 'knn', 'dt', 'soft voting'])

pred_df

,0,1,2,3,4,5,6,7,8,9
true,1.000000,0.000000,1.000000,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000
lr,0.989179,0.001332,0.998704,0.000026,0.000276,0.901632,0.365862,0.834998,0.999966,0.983017
knn,1.000000,0.000000,1.000000,0.000000,0.000000,0.800000,0.600000,0.600000,1.000000,0.600000
dt,1.000000,0.000000,1.000000,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000
soft voting,0.996393,0.000444,0.999568,0.000009,0.000092,0.900544,0.655287,0.811666,0.999989,0.861006


## 2. VotingRegressor 회귀 예제

회귀에서는 클래스를 투표하는 대신, 여러 모델의 예측값을 평균내는 방식으로 결합한다.

즉, 분류의 soft voting과 비슷하게 생각할 수 있지만,
최종 출력이 확률이 아니라 연속형 수치라는 점이 다르다.


In [12]:
# 캘리포니아 주택 가격 데이터 로드
df = pd.read_csv('data/california_housing.csv')

# 데이터 구조 확인
print(df.head())
print(df.info())

# 타겟과 피처 분리
X = df.drop('MedHouseVal', axis=1).to_numpy()
y = df['MedHouseVal'].to_numpy()

# 훈련/테스트 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 회귀에서도 KNN은 거리 기반이라 스케일 영향을 받는다.
# 선형회귀 역시 스케일링을 하면 계수 해석 외에는 계산 안정성 측면에서 도움이 될 수 있다.
# 트리는 꼭 필요하지 않지만, 입력 일관성을 위해 함께 사용한다.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(X_train_scaled.shape, y_train.shape)
print(X_test_scaled.shape, y_test.shape)

   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3  5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
4  3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   

   Longitude  MedHouseVal  
0    -122.23        4.526  
1    -122.22        3.585  
2    -122.24        3.521  
3    -122.25        3.413  
4    -122.25        3.422  
<class 'pandas.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   MedInc       20640 non-null  float64
 1   HouseAge     20640 non-null  float64
 2   AveRooms     20640 non-null  float64
 3   AveBedrms    20640 non-null  float64
 4   Population   20640 no

In [14]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import VotingRegressor

# 개별 회귀기 준비
lin_reg = LinearRegression()
knn_reg = KNeighborsRegressor(n_neighbors=5)
dt_reg = DecisionTreeRegressor(random_state=42)

# Voting 회귀기 준비
voting_reg = VotingRegressor(
    estimators=[
        ('lin', lin_reg),
        ('knn', knn_reg),
        ('dt', dt_reg),
    ]
)

# 앙상블 학습
voting_reg.fit(X_train_scaled, y_train)


,"estimators estimators: list of (str, estimator) tuplesInvoking the ``fit`` method on the ``VotingRegressor`` will fit clonesof those original estimators that will be stored in the class attribute``self.estimators_``. An estimator can be set to ``'drop'`` using:meth:`set_params`... versionchanged:: 0.21 ``'drop'`` is accepted. Using None was deprecated in 0.22 and support was removed in 0.24.","[('lin', ...), ('knn', ...), ...]"
,"weights weights: array-like of shape (n_regressors,), default=NoneSequence of weights (`float` or `int`) to weight the occurrences ofpredicted values before averaging. Uses uniform weights if `None`.",None
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for ``fit``.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting will be printed as itis completed... versionadded:: 0.23",False
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Uniform weights are used by default.See the following example for a demonstration of the impact ofdifferent weighting schemes on predictions::ref:`sphx_glr_auto_examples_neighbors_plot_regression.py`.",'uniform'


In [15]:
# 개별 회귀기와 VotingRegressor의 성능 비교
from sklearn.metrics import r2_score, root_mean_squared_error

lin_reg.fit(X_train_scaled, y_train)
knn_reg.fit(X_train_scaled, y_train)
dt_reg.fit(X_train_scaled, y_train)

result = []

for reg in [lin_reg, knn_reg, dt_reg, voting_reg]:
    model_name = reg.__class__.__name__
    y_pred = reg.predict(X_test_scaled)

    result.append({
        'model' : model_name,
        'r^2' : r2_score(y_test, y_pred),
        'RMSE' : root_mean_squared_error(y_test, y_pred)
    })

pd.DataFrame(result).sort_values('RMSE')

,model,r^2,RMSE
3,VotingRegressor,0.732867,0.591653
1,KNeighborsRegressor,0.670010,0.657588
2,DecisionTreeRegressor,0.623042,0.702829
0,LinearRegression,0.575788,0.745581


In [16]:
# 일부 샘플에서 예측 값 평균이 어떻게 만들어 지는지 확인
lin_pred = lin_reg.predict(X_train_scaled)
knn_pred = knn_reg.predict(X_train_scaled)
dt_pred = dt_reg.predict(X_train_scaled)
voting_reg_pred = voting_reg.predict(X_train_scaled)

start = 0
end = 10

pred_df = pd.DataFrame({
    'true' : y_test[start:end],
    'lin' : lin_pred[start:end],
    'knn' : knn_pred[start:end],
    'dt' : dt_pred[start:end],
    'voting' : voting_reg_pred[start:end],
})

pred_df


,true,lin,knn,dt,voting
0,0.47700,1.937258,1.238200,1.03000,1.401819
1,0.45800,2.489106,2.868400,3.82100,3.059502
2,5.00001,2.647355,1.787600,1.72600,2.053652
3,2.18600,1.565895,0.973200,0.93400,1.157698
4,2.78000,1.613128,0.876400,0.96500,1.151509
5,1.58700,3.283596,2.776400,2.64800,2.902665
6,1.98200,1.544837,1.675600,1.57300,1.597812
7,1.57500,4.139530,4.081204,5.00001,4.406915
8,3.40000,0.843304,1.640000,1.39800,1.293768
9,4.46600,2.659941,3.041200,3.15600,2.952380


## 정리

1. 앙상블은 여러 모델의 장점을 결합하려는 전략이다.
2. Hard Voting은 다수결, Soft Voting은 확률 평균이다.
3. 회귀 앙상블은 예측값 평균으로 이해하면 된다.
4. 앙상블 성능은 반드시 개별 모델과 비교해서 판단해야 한다.
5. 서로 다른 성향의 모델을 섞는 것이 앙상블의 핵심 아이디어이다.